In [6]:
import os
import numpy as np
from PIL import Image

path_to_png_train = "/home/pablo.canosa/ssd/datasets_pablo/rios_FBP_balanced/png"
path_to_png_test = "/home/pablo.canosa/ssd/datasets_pablo/rios_FBP_balanced/png_test"

# Mapping list (Index 0 = ID 1, Index 1 = ID 2, etc.)
class_names = [
    "Water",        # 1
    "Bare soil",    # 2
    "Rock",         # 3
    "Asphalt",      # 4
    "Concrete",     # 5
    "Tiles",        # 6
    "Meadows",      # 7
    "Native trees", # 8
    "Pines",        # 9
    "Eucalyptus"    # 10
]

def get_class_name(class_id):
    if class_id == 0:
        return "Background/Unclassified"
    # Subtract 1 because list indices start at 0
    try:
        return class_names[class_id - 1]
    except IndexError:
        return f"Unknown({class_id})"

def count_pixels(path):
    class_counts = {}
    if not os.path.exists(path):
        print(f"Warning: Path {path} not found.")
        return class_counts
        
    for filename in os.listdir(path):
        if filename.lower().endswith(".png"):
            image_path = os.path.join(path, filename)
            with Image.open(image_path) as img:
                image = np.array(img)
                unique, counts = np.unique(image, return_counts=True)
                for u, c in zip(unique, counts):
                    class_counts[u] = class_counts.get(u, 0) + int(c)
    return class_counts

# Execute counting
train_raw = count_pixels(path_to_png_train)
test_raw = count_pixels(path_to_png_test)

# Combine raw counts first
all_ids = set(train_raw.keys()) | set(test_raw.keys())
whole_raw = {k: train_raw.get(k, 0) + test_raw.get(k, 0) for k in all_ids}

# 1. Create named counts for printing
train_counts = {get_class_name(k): v for k, v in train_raw.items()}
test_counts = {get_class_name(k): v for k, v in test_raw.items()}
whole_counts = {get_class_name(k): v for k, v in whole_raw.items()}

print("Class counts in training set:", train_counts)
print("Class counts in test set:", test_counts)
print("Total counts:", whole_counts)

# 2. Calculate distribution excluding class 0
total_pixels_ex_0 = sum(v for k, v in whole_raw.items() if k != 0)

if total_pixels_ex_0 > 0:
    # Map the distribution to names
    class_distribution = {get_class_name(k): v / total_pixels_ex_0 
                          for k, v in whole_raw.items() if k != 0}
    
    print("\nClass distribution (excluding class 0):")
    for name, percentage in class_distribution.items():
        print(f"{name}: {percentage:.4%}")
else:
    print("No pixels found for classes other than 0.")

Class counts in training set: {'Background/Unclassified': 243039206, 'Native trees': 9124744, 'Eucalyptus': 16057423, 'Bare soil': 7586328, 'Meadows': 28167394, 'Water': 1100236, 'Rock': 744841, 'Asphalt': 1029493, 'Concrete': 153126, 'Tiles': 158561, 'Pines': 338648}
Class counts in test set: {'Background/Unclassified': 43062839, 'Bare soil': 1353733, 'Meadows': 4188004, 'Native trees': 1376403, 'Eucalyptus': 2363802, 'Asphalt': 159072, 'Pines': 153663, 'Rock': 118155, 'Water': 166492, 'Concrete': 7523, 'Tiles': 50314}
Total counts: {'Background/Unclassified': 286102045, 'Water': 1266728, 'Bare soil': 8940061, 'Rock': 862996, 'Asphalt': 1188565, 'Concrete': 160649, 'Tiles': 208875, 'Meadows': 32355398, 'Native trees': 10501147, 'Pines': 492311, 'Eucalyptus': 18421225}

Class distribution (excluding class 0):
Water: 1.7026%
Bare soil: 12.0165%
Rock: 1.1600%
Asphalt: 1.5976%
Concrete: 0.2159%
Tiles: 0.2808%
Meadows: 43.4896%
Native trees: 14.1148%
Pines: 0.6617%
Eucalyptus: 24.7604%
